# 🧠 RAG-Based Profile Matching System
## A Complete AI Engineering Workbook

---

### 📚 Learning Objectives
By the end of this workbook, you will be able to:
1. **Implement document chunking** — intelligently split resumes preserving semantic sections
2. **Generate embeddings** — use Cohere/OpenAI/HuggingFace embedding models to encode text into vector space
3. **Build a vector database** — store and index embeddings with ChromaDB
4. **Create a retrieval pipeline** — semantic search + BM25 hybrid retrieval
5. **Understand semantic search** — measure similarity in high-dimensional embedding space
6. **Rank and score candidates** — produce explainable match scores (0–100)

---

### 🗂️ Workbook Structure

| Section | Topic | Part |
|---------|-------|------|
| 0 | Environment Setup & Dependencies | Setup |
| 1 | Document Loading Pipeline | Part A (50%) |
| 2 | Intelligent Chunking Strategy | Part A (50%) |
| 3 | Embedding Generation (Cohere) | Part A (50%) |
| 4 | ChromaDB Vector Database | Part A (50%) |
| 5 | Metadata Extraction | Part A (50%) |
| 6 | Semantic Search Pipeline | Part B (50%) |
| 7 | Hybrid Search (Semantic + BM25) | Part B (50%) |
| 8 | Candidate Scoring & Ranking | Part B (50%) |
| 9 | Structured JSON Output | Part B (50%) |
| 10 | End-to-End Evaluation & Demo | Full Pipeline |

---

> **🔑 API Keys Required:** Cohere API key (or free local SentenceTransformer fallback)
> 
> **📦 Install dependencies** by running Section 0 first.

---
## Section 0: Environment Setup & Dependencies

### 🎯 Why This Matters
A production-grade RAG pipeline requires specialized tools:
- **`chromadb`** — local, persistent vector database
- **`cohere` / `sentence-transformers`** — embedding models
- **`pypdf`** — PDF document parsing
- **`rank_bm25`** — BM25 keyword retrieval engine
- **`rapidfuzz`** — fuzzy skill alias matching
- **`python-dotenv`** — configuration management

In [ ]:
# ─────────────────────────────────────────────────────────────
# Section 0.1 — Install Required Packages
# ─────────────────────────────────────────────────────────────
import subprocess, sys

packages = [
    "chromadb",
    "cohere",
    "pypdf",
    "rank_bm25",
    "python-dotenv",
    "sentence-transformers",
    "pandas",
    "numpy",
    "rapidfuzz",
    "pydantic",
    "tqdm",
    "matplotlib",
    "seaborn",
]

for pkg in packages:
    print(f"Installing {pkg}...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "-q"],
        capture_output=True
    )

print("\n✅ All required packages installed!")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Section 0.2 — Imports & Environment Config
# ─────────────────────────────────────────────────────────────
import os
import re
import json
import time
import hashlib
import warnings
import pathlib
from typing import List, Dict, Optional, Tuple, Any
from dataclasses import dataclass, field, asdict
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from rank_bm25 import BM25Okapi
from rapidfuzz import fuzz
from pypdf import PdfReader
import chromadb
import cohere
from dotenv import load_dotenv
from IPython.display import display, Markdown, JSON

warnings.filterwarnings('ignore')
load_dotenv()
load_dotenv(dotenv_path="../.env")

# Global Configuration
CONFIG = {
    "RESUMES_DIR": "resumes",
    "CHROMA_DIR": "chroma_db",
    "COLLECTION_NAME": "resumes_v1",
    "CHUNK_SIZE": 800,
    "CHUNK_OVERLAP": 120,
    "TOP_K": 10,
    "HYBRID_ALPHA": 0.65,
    "COHERE_MODEL": "embed-english-v3.0",
    "SCORE_WEIGHTS": {
        "semantic_similarity": 0.40,
        "skill_overlap": 0.30,
        "experience_match": 0.20,
        "keyword_density": 0.10,
    }
}

COHERE_API_KEY = os.getenv("COHERE_API_KEY", "")
print(f"✅ System setup complete. Cohere API key present: {bool(COHERE_API_KEY)}")

---
## Section 1: Document Processing Pipeline (Part A)

### 🎯 Load Resumes from File System
We create a dataset of sample resumes in `./resumes/` representing diverse software & AI candidate profiles, then read and normalize them using file system tools.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Section 1.1 — Synthetic Resumes Generator
# ─────────────────────────────────────────────────────────────
SYNTHETIC_RESUMES = [
    {
        "filename": "resumes/john_doe.txt",
        "content": """JOHN DOE
Email: john.doe@email.com | Phone: +1-555-0101
Location: San Francisco, CA

PROFESSIONAL SUMMARY
Senior Machine Learning Engineer with 7 years of experience building production ML & NLP systems.

SKILLS
Programming: Python, Scala, SQL, Bash
ML/DL: PyTorch, TensorFlow, Scikit-learn, Hugging Face Transformers
NLP/RAG: BERT, GPT, RAG, LangChain, Pinecone, ChromaDB, Vector Databases
MLOps: MLflow, Kubeflow, Airflow, Docker, Kubernetes, AWS SageMaker

PROFESSIONAL EXPERIENCE
Senior ML Engineer | TechCorp Inc. | 2020 – Present (4 years)
• Built RAG-based document AI chatbot reducing ticket volume by 40%
• Led real-time fraud detection service processing 2M transactions/day with PyTorch
• Architected MLOps deployment pipeline using Docker and Kubernetes

ML Engineer | DataWave Analytics | 2017 – 2020 (3 years)
• Developed sentiment analysis models with 94% accuracy
• Built recommendation engine using Scikit-learn

EDUCATION
M.S. Computer Science | Stanford University | 2017
B.S. Mathematics | UC Berkeley | 2015
"""
    },
    {
        "filename": "resumes/jane_smith.txt",
        "content": """JANE SMITH
Email: jane.smith@email.com | Phone: +1-555-0102
Location: New York, NY

PROFESSIONAL SUMMARY
Data Scientist with 5 years specializing in NLP, statistical modeling, and predictive analytics.

SKILLS
Languages: Python, R, SQL
ML: PyTorch, XGBoost, LightGBM, Scikit-learn
NLP: spaCy, NLTK, Transformers, OpenAI API, LangChain
Data: Pandas, NumPy, Spark, Databricks

PROFESSIONAL EXPERIENCE
Data Scientist II | FinanceAI Corp | 2021 – Present (3 years)
• Built financial entity extraction NLP pipeline with Transformers
• Developed credit risk prediction model reducing default rates by 18%

Data Scientist | HealthTech Solutions | 2019 – 2021 (2 years)
• Built patient readmission prediction model with 87% AUC

EDUCATION
M.S. Statistics | Columbia University | 2019
B.S. Computer Science | NYU | 2017
"""
    },
    {
        "filename": "resumes/alex_kumar.txt",
        "content": """ALEX KUMAR
Email: alex.kumar@email.com | Phone: +1-555-0103
Location: Seattle, WA

PROFESSIONAL SUMMARY
MLOps & Infrastructure Engineer with 6 years experience designing Kubernetes-native ML platforms.

SKILLS
Infrastructure: Kubernetes, Docker, Terraform, Ansible
ML Platforms: MLflow, Kubeflow, Ray, Airflow
Languages: Python, Go, Bash
Cloud: AWS, GCP

PROFESSIONAL EXPERIENCE
Senior MLOps Engineer | CloudScale ML | 2021 – Present (3 years)
• Built Kubernetes-native training platform serving 200+ data scientists
• Reduced ML infrastructure costs by 60% with spot instance optimization

DevOps Engineer | NextGen AI | 2018 – 2021 (3 years)
• Migrated ML workloads to Kubernetes cluster

EDUCATION
B.S. Computer Engineering | University of Washington | 2018
"""
    },
    {
        "filename": "resumes/sarah_johnson.txt",
        "content": """SARAH JOHNSON
Email: sarah.j@email.com | Phone: +1-555-0106
Location: Chicago, IL

PROFESSIONAL SUMMARY
Backend Software Engineer with 5 years building high-throughput Python and Go microservices.

SKILLS
Languages: Python, Go, SQL, Java
Frameworks: FastAPI, gRPC, Django
Data/Cloud: Kafka, PostgreSQL, Redis, AWS

PROFESSIONAL EXPERIENCE
Senior Backend Engineer | StreamTech | 2021 – Present (3 years)
• Built Python FastAPI microservices handling 10M events/day
• Reduced API latency from 200ms to 15ms

Software Engineer | PaymentCo | 2019 – 2021 (2 years)
• Developed PCI-compliant payment gateway in Python

EDUCATION
B.S. Computer Science | University of Illinois | 2019
"""
    },
    {
        "filename": "resumes/ryan_thompson.txt",
        "content": """RYAN THOMPSON
Email: ryan.t@email.com | Phone: +1-555-0109
Location: Denver, CO

PROFESSIONAL SUMMARY
Junior ML Engineer with 2 years experience in Python machine learning and data analysis.

SKILLS
Languages: Python, SQL
ML: Scikit-learn, Pandas, NumPy, Matplotlib
DL: PyTorch (beginner)
Tools: Git, Docker, Jupyter

PROFESSIONAL EXPERIENCE
Junior ML Engineer | SmallStartup | 2022 – Present (2 years)
• Built customer churn prediction model using Scikit-learn (78% accuracy)
• Automated reporting ETL with Pandas

EDUCATION
B.S. Data Science | University of Colorado | 2021
"""
    }
]

os.makedirs("resumes", exist_ok=True)
for r in SYNTHETIC_RESUMES:
    with open(r["filename"], "w", encoding="utf-8") as f:
        f.write(r["content"])

print(f"✅ Wrote {len(SYNTHETIC_RESUMES)} sample resumes to ./resumes/")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Section 1.2 — Document Loader
# ─────────────────────────────────────────────────────────────
@dataclass
class Document:
    doc_id: str
    file_path: str
    filename: str
    content: str
    format: str

class DocumentLoader:
    def load_directory(self, directory: str) -> List[Document]:
        docs = []
        path = pathlib.Path(directory)
        for file_path in sorted(list(path.glob("*.txt")) + list(path.glob("*.pdf"))):
            ext = file_path.suffix.lower()
            if ext == ".txt":
                with open(file_path, "r", encoding="utf-8") as f:
                    content = f.read()
            elif ext == ".pdf":
                reader = PdfReader(file_path)
                content = "\n".join([p.extract_text() for p in reader.pages if p.extract_text()])
            else:
                continue
            
            doc_id = hashlib.md5(file_path.name.encode()).hexdigest()[:12]
            docs.append(Document(
                doc_id=doc_id,
                file_path=str(file_path),
                filename=file_path.name,
                content=content.strip(),
                format=ext.lstrip('.')
            ))
        return docs

loader = DocumentLoader()
documents = loader.load_directory(CONFIG["RESUMES_DIR"])
print(f"📂 Loaded {len(documents)} document(s) from filesystem.")

---
## Section 2: Intelligent Chunking Strategy

### 🎯 Preserve Resume Sections
Instead of blind fixed-size slicing, we detect section headers like `SKILLS`, `EXPERIENCE`, `EDUCATION`, and `SUMMARY` to chunk intelligently.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Section 2.1 — Section-Aware Resume Chunker
# ─────────────────────────────────────────────────────────────
@dataclass
class Chunk:
    chunk_id: str
    doc_id: str
    file_path: str
    filename: str
    section: str
    text: str
    chunk_idx: int

class IntelligentResumeChunker:
    SECTION_PATTERNS = {
        "SUMMARY": re.compile(r'^(PROFESSIONAL\s+SUMMARY|SUMMARY|PROFILE)', re.I | re.M),
        "SKILLS": re.compile(r'^(TECHNICAL\s+SKILLS|SKILLS|COMPETENCIES)', re.I | re.M),
        "EXPERIENCE": re.compile(r'^(PROFESSIONAL\s+EXPERIENCE|WORK\s+EXPERIENCE|EXPERIENCE)', re.I | re.M),
        "EDUCATION": re.compile(r'^(EDUCATION|ACADEMIC\s+BACKGROUND)', re.I | re.M),
    }

    def chunk_document(self, doc: Document) -> List[Chunk]:
        text = doc.content
        matches = []
        for sec, pat in self.SECTION_PATTERNS.items():
            for m in pat.finditer(text):
                matches.append((m.start(), sec))
        matches.sort(key=lambda x: x[0])

        chunks = []
        chunk_idx = 0
        if not matches:
            # Fallback to full doc chunk
            chunks.append(Chunk(f"{doc.doc_id}_0", doc.doc_id, doc.file_path, doc.filename, "FULL", text, 0))
            return chunks

        # Header
        if matches[0][0] > 0:
            chunks.append(Chunk(f"{doc.doc_id}_{chunk_idx}", doc.doc_id, doc.file_path, doc.filename, "HEADER", text[:matches[0][0]].strip(), chunk_idx))
            chunk_idx += 1

        for i, (pos, sec_name) in enumerate(matches):
            end_pos = matches[i+1][0] if i+1 < len(matches) else len(text)
            sec_text = text[pos:end_pos].strip()
            if sec_text:
                chunks.append(Chunk(f"{doc.doc_id}_{chunk_idx}", doc.doc_id, doc.file_path, doc.filename, sec_name, sec_text, chunk_idx))
                chunk_idx += 1
        return chunks

chunker = IntelligentResumeChunker()
all_chunks = []
for doc in documents:
    all_chunks.extend(chunker.chunk_document(doc))

print(f"✅ Created {len(all_chunks)} intelligent section chunks across {len(documents)} resumes.")

---
## Section 3: Embeddings Generation & Vector Database (Part A)

### 🎯 Vector Storage with ChromaDB
Generate text embeddings using Cohere `embed-english-v3.0` (or local fallback) and store them in ChromaDB alongside section metadata.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Section 3.1 — Embedding Service & Vector DB Setup
# ─────────────────────────────────────────────────────────────
class EmbeddingService:
    def __init__(self, api_key: str):
        self.use_cohere = False
        if api_key:
            try:
                self.client = cohere.ClientV2(api_key=api_key)
                self.use_cohere = True
            except Exception as e:
                print(f"⚠️ Cohere init failed, using local model: {e}")
        if not self.use_cohere:
            from sentence_transformers import SentenceTransformer
            self.local_model = SentenceTransformer("all-MiniLM-L6-v2")

    def embed_texts(self, texts: List[str], is_query: bool = False) -> List[List[float]]:
        if self.use_cohere:
            input_type = "search_query" if is_query else "search_document"
            res = self.client.embed(texts=texts, model="embed-english-v3.0", input_type=input_type, embedding_types=["float"])
            return res.embeddings.float_
        else:
            return self.local_model.encode(texts).tolist()

embedder = EmbeddingService(COHERE_API_KEY)
embeddings = embedder.embed_texts([c.text for c in all_chunks])

# Setup ChromaDB
chroma_client = chromadb.PersistentClient(path=CONFIG["CHROMA_DIR"])
collection = chroma_client.get_or_create_collection(name=CONFIG["COLLECTION_NAME"], metadata={"hnsw:space": "cosine"})

collection.upsert(
    ids=[c.chunk_id for c in all_chunks],
    embeddings=embeddings,
    documents=[c.text for c in all_chunks],
    metadatas=[{"doc_id": c.doc_id, "filename": c.filename, "file_path": c.file_path, "section": c.section} for c in all_chunks]
)

print(f"✅ Stored {collection.count()} vector embeddings in ChromaDB persistent collection.")

---
## Section 4: Metadata Extraction (Part A)

### 🎯 Extract Key Fields
Extract Name, Skills, Experience Years, Education for structured filtering & scoring.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Section 4.1 — Metadata Extraction Engine
# ─────────────────────────────────────────────────────────────
@dataclass
class ResumeMetadata:
    doc_id: str
    candidate_name: str
    resume_path: str
    skills: List[str]
    years_experience: float
    education: str
    raw_text: str

SKILLS_DB = ["Python", "Scala", "SQL", "PyTorch", "TensorFlow", "Scikit-learn", "Hugging Face", "RAG", "LangChain", "Pinecone", "ChromaDB", "MLflow", "Kubeflow", "Airflow", "Docker", "Kubernetes", "AWS", "GCP", "R", "spaCy", "NLTK", "OpenAI", "Spark", "Databricks", "Terraform", "FastAPI", "Django", "Kafka", "PostgreSQL", "Redis"]

class MetadataExtractor:
    def extract(self, doc: Document) -> ResumeMetadata:
        text = doc.content
        name = text.split('\n')[0].title()
        
        # Skills
        found_skills = [s for s in SKILLS_DB if re.search(r'\b' + re.escape(s) + r'\b', text, re.I)]
        
        # Years
        years_match = re.findall(r'\((\d+)\s+years?\)', text, re.I)
        years = sum(float(y) for y in years_match) if years_match else 2.0
        
        # Education
        edu_match = re.search(r'(M\.S\.|B\.S\.|Ph\.D\.|Bachelor|Master)[^\n]+', text)
        edu = edu_match.group(0) if edu_match else "Degree"
        
        return ResumeMetadata(doc.doc_id, name, doc.file_path, found_skills, years, edu, text)

extractor = MetadataExtractor()
metadata_store = {doc.doc_id: extractor.extract(doc) for doc in documents}

print("✅ Extracted metadata for candidates:")
for m in metadata_store.values():
    print(f"  • {m.candidate_name}: {m.years_experience} yrs exp | Skills: {m.skills[:4]}")

---
## Section 5: Job Matching Engine & Hybrid Search (Part B)

### 🎯 Hybrid Retrieval + Ranking & Scoring
- **Semantic Search:** Convert JD to embedding, retrieve top chunks.
- **Hybrid Search:** Combine Semantic vector similarity + BM25 keyword matching.
- **Must-Have Filtering:** Filter candidates by minimum experience or mandatory skills.
- **0-100 Scoring & Reasoning:** Produce detailed match output.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Section 5.1 — Hybrid Search & Job Matching Engine
# ─────────────────────────────────────────────────────────────
class JobMatchingEngine:
    def __init__(self, collection, embedder, metadata_store, chunks):
        self.collection = collection
        self.embedder = embedder
        self.metadata_store = metadata_store
        self.chunks = chunks
        
        # BM25
        corpus = [c.text.lower().split() for c in chunks]
        self.bm25 = BM25Okapi(corpus)

    def match(
        self,
        job_description: str,
        min_years_exp: float = 0.0,
        critical_skills: List[str] = None,
        top_k: int = 10
    ) -> Dict[str, Any]:
        critical_skills = critical_skills or []
        
        # 1. Semantic Search
        jd_emb = self.embedder.embed_texts([job_description], is_query=True)[0]
        vector_res = self.collection.query(query_embeddings=[jd_emb], n_results=len(self.chunks))
        
        sem_scores = {}
        for i, chunk_id in enumerate(vector_res["ids"][0]):
            doc_id = vector_res["metadatas"][0][i]["doc_id"]
            dist = vector_res["distances"][0][i]
            sim = max(0.0, 1.0 - dist)
            sem_scores[doc_id] = max(sem_scores.get(doc_id, 0), sim)
            
        # 2. BM25 Search
        bm25_scores_raw = self.bm25.get_scores(job_description.lower().split())
        max_bm25 = max(bm25_scores_raw) if len(bm25_scores_raw) > 0 and max(bm25_scores_raw) > 0 else 1.0
        
        bm25_doc_scores = {}
        for idx, score in enumerate(bm25_scores_raw):
            doc_id = self.chunks[idx].doc_id
            bm25_doc_scores[doc_id] = max(bm25_doc_scores.get(doc_id, 0), score / max_bm25)
            
        # 3. Match & Filter Candidates
        matches = []
        for doc_id, meta in self.metadata_store.items():
            # Filter: Must-have experience
            if meta.years_experience < min_years_exp:
                continue
                
            # Filter: Critical skills
            matched_skills = [s for s in SKILLS_DB if re.search(r'\b' + re.escape(s) + r'\b', job_description, re.I) and s in meta.skills]
            missing_critical = [cs for cs in critical_skills if cs not in meta.skills]
            if missing_critical:
                continue
                
            # Hybrid scoring
            sem_val = sem_scores.get(doc_id, 0.5)
            bm25_val = bm25_doc_scores.get(doc_id, 0.0)
            hybrid_score = 0.65 * sem_val + 0.35 * bm25_val
            
            # Final score scale 0-100
            final_score = int(round(hybrid_score * 100))
            
            # Reasoning & Excerpts
            excerpts = [c.text for c in self.chunks if c.doc_id == doc_id and c.section in ["SKILLS", "EXPERIENCE"]][:2]
            reasoning = f"Strong match for experience ({meta.years_experience} yrs) with key skills: {', '.join(matched_skills[:4])}."
            
            matches.append({
                "candidate_name": meta.candidate_name,
                "resume_path": meta.resume_path,
                "match_score": final_score,
                "matched_skills": matched_skills,
                "relevant_excerpts": excerpts,
                "reasoning": reasoning
            })
            
        matches.sort(key=lambda x: x["match_score"], reverse=True)
        
        return {
            "job_description": job_description,
            "top_matches": matches[:top_k]
        }

engine = JobMatchingEngine(collection, embedder, metadata_store, all_chunks)
print("✅ Job Matching Engine ready.")

---
## Section 6: End-to-End Evaluation & JSON Output (Part B)

### 🎯 Sample Job Match Test
Run a test query with a sample Job Description and output the exact required JSON structure.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Section 6.1 — Execute Job Match & Display Required JSON Output
# ─────────────────────────────────────────────────────────────
sample_jd = """
We are looking for a Senior Machine Learning Engineer with 5+ years of experience.
Must have deep expertise in Python, PyTorch, RAG, and Vector Databases.
Experience with Docker and Kubernetes for MLOps is preferred.
"""

result = engine.match(
    job_description=sample_jd.strip(),
    min_years_exp=3.0,
    critical_skills=["Python"],
    top_k=10
)

# Print formatted JSON as per assignment requirements
print("=== REQUIRED OUTPUT FORMAT ===\n")
print(json.dumps(result, indent=2))

# Save output to file
with open("match_results.json", "w") as f:
    json.dump(result, f, indent=2)
print("\n✅ Match results saved to match_results.json")